# S2 — LABORATORIO EN CASA
## Big Data DD283 | Universidad Autónoma del Perú | 2026-1
### Semana 2: Procesamiento Distribuido con PySpark en Databricks

---

**Nombre del estudiante**:  Tolentino Vargas Jesus Antonio
**Fecha de entrega**: 18/06/2026  
**Modalidad**: Individual

---

### Objetivo
Implementar un pipeline Big Data completo con PySpark: desde la carga de datos hasta el análisis distribuido con operaciones equivalentes a MapReduce (groupBy, aggregations) y generación de insights sobre un dataset de accidentes de tránsito del MTC Perú.

> **Nota de ejecución:** Este notebook está preparado para correr tanto en **Databricks Community Edition** como en **Google Colab**. La celda de configuración detecta el entorno automáticamente. En Databricks, omite la celda de instalación de PySpark (Spark ya está disponible).


## CONFIGURACIÓN DEL ENTORNO

### Verificación (Databricks) / Instalación (Colab)

In [1]:
# === SI ESTÁS EN GOOGLE COLAB: ejecuta esta celda ===
# === SI ESTÁS EN DATABRICKS: OMITE esta celda (Spark ya existe) ===

import sys

# Detectar si Spark ya existe (Databricks) o hay que crearlo (Colab)
try:
    spark
    print("Entorno Databricks detectado: Spark ya está disponible.")
except NameError:
    print("Entorno Colab detectado: instalando PySpark...")
    import subprocess
    subprocess.run(["pip", "install", "pyspark==3.5.0", "--quiet"])
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("BigDataS2Lab") \
        .master("local[2]") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()
    sc = spark.sparkContext

print(f"Spark version: {spark.version}")
print("Entorno listo!")

Entorno Colab detectado: instalando PySpark...
Spark version: 3.5.0
Entorno listo!


## DATOS DEL LABORATORIO

Dataset sintético de accidentes de tránsito basado en estadísticas reales del MTC Perú 2023 (50,000 registros).

In [2]:
# Generar dataset sintético basado en estadísticas reales del MTC Perú 2023
import pandas as pd
import numpy as np

np.random.seed(42)
n = 50000  # 50,000 registros de accidentes

departamentos = ['Lima', 'Arequipa', 'La Libertad', 'Piura', 'Cusco',
                 'Junin', 'Puno', 'Ancash', 'Loreto', 'Cajamarca']
tipos = ['Choque', 'Atropello', 'Volcadura', 'Caida_ocupante', 'Incendio', 'Despiste']
causas = ['Exceso_velocidad', 'Impericia', 'Embriaguez', 'Exceso_carga',
          'Falla_mecanica', 'Mal_estado_via', 'Distraccion', 'No_respeta_senal']
meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
         'Julio','Agosto','Setiembre','Octubre','Noviembre','Diciembre']

probs_dep = [0.35, 0.12, 0.10, 0.08, 0.07, 0.07, 0.05, 0.05, 0.06, 0.05]

data = {
    'ID_accidente': range(1, n+1),
    'departamento': np.random.choice(departamentos, n, p=probs_dep),
    'tipo_accidente': np.random.choice(tipos, n, p=[0.45, 0.20, 0.15, 0.10, 0.02, 0.08]),
    'causa_principal': np.random.choice(causas, n),
    'mes': np.random.choice(meses, n),
    'hora': np.random.randint(0, 24, n),
    'fallecidos': np.random.choice([0,1,2,3], n, p=[0.85, 0.10, 0.03, 0.02]),
    'heridos': np.random.randint(0, 8, n),
    'vehiculos_involucrados': np.random.randint(1, 5, n),
    'danio_material_soles': np.random.exponential(5000, n).astype(int)
}

df_pandas = pd.DataFrame(data)
df_pandas.to_csv('/tmp/accidentes_mtc_peru.csv', index=False)
print(f"Dataset creado: {len(df_pandas):,} registros")
print(df_pandas.head(3))

Dataset creado: 50,000 registros
   ID_accidente departamento  tipo_accidente   causa_principal     mes  hora  \
0             1     Arequipa  Caida_ocupante         Impericia  Agosto    18   
1             2    Cajamarca       Atropello  No_respeta_senal    Mayo    16   
2             3        Junin          Choque    Mal_estado_via  Agosto    15   

   fallecidos  heridos  vehiculos_involucrados  danio_material_soles  
0           0        2                       1                  2819  
1           0        4                       1                  3384  
2           0        2                       2                  3781  


## PARTE 1 — EXPLORACIÓN DEL ENTORNO SPARK

### Actividad 1.1 — Cargar datos en Spark

In [3]:
# Cargar el CSV en un DataFrame de Spark
df = spark.read.csv('/tmp/accidentes_mtc_peru.csv',
                    header=True,       # Primera fila = nombres de columnas
                    inferSchema=True)  # Detecta automáticamente int, string, etc.

# Ver la estructura del DataFrame (equivale a DESCRIBE TABLE en SQL)
df.printSchema()

# Ver las primeras 5 filas
df.show(5)

# Contar registros (equivale a SELECT COUNT(*) FROM tabla)
print(f"\nTotal de registros: {df.count():,}")

root
 |-- ID_accidente: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- tipo_accidente: string (nullable = true)
 |-- causa_principal: string (nullable = true)
 |-- mes: string (nullable = true)
 |-- hora: integer (nullable = true)
 |-- fallecidos: integer (nullable = true)
 |-- heridos: integer (nullable = true)
 |-- vehiculos_involucrados: integer (nullable = true)
 |-- danio_material_soles: integer (nullable = true)

+------------+------------+--------------+----------------+---------+----+----------+-------+----------------------+--------------------+
|ID_accidente|departamento|tipo_accidente| causa_principal|      mes|hora|fallecidos|heridos|vehiculos_involucrados|danio_material_soles|
+------------+------------+--------------+----------------+---------+----+----------+-------+----------------------+--------------------+
|           1|    Arequipa|Caida_ocupante|       Impericia|   Agosto|  18|         0|      2|                     1|                281

### Actividad 1.2 — Explorar el SparkUI

In [4]:
# Ejecuta esta consulta y LUEGO mira el SparkUI:
resultado_prueba = df.groupBy("departamento").count()
resultado_prueba.show()

# En SparkUI (Databricks: "Spark Jobs" arriba | Colab: http://localhost:4040) verás:
# - Stages: cuántas etapas tuvo el job
# - Tasks: cuántas tareas corrieron en paralelo
# - Shuffle Read/Write: cuántos datos se movieron entre nodos

+------------+-----+
|departamento|count|
+------------+-----+
|   Cajamarca| 2470|
|    Arequipa| 5953|
|        Lima|17608|
|       Cusco| 3525|
|      Ancash| 2469|
|       Junin| 3474|
|        Puno| 2458|
|       Piura| 4053|
| La Libertad| 4984|
|      Loreto| 3006|
+------------+-----+



**Pregunta de reflexión 1.1 — ¿Cuántas stages tiene el job de groupBy? ¿Cuál es la etapa Map y cuál la Reduce?**

> **Respuesta:** El job de `groupBy` tiene **2 stages**.
> - **Stage 1 (equivalente a Map):** Spark lee el CSV y hace un conteo parcial local en cada partición (pre-agregación local por departamento). Esto corresponde a la fase Map de MapReduce.
> - **Stage 2 (equivalente a Reduce):** ocurre después del *shuffle*. Spark redistribuye los datos por la clave (departamento) para que todos los registros de un mismo departamento queden juntos, y suma los conteos parciales. Esto corresponde a la fase Reduce.
>
> El límite entre ambas stages es exactamente el **shuffle**: el momento en que los datos se mueven entre nodos para agruparse por clave.


### Actividad 1.3 — Operaciones básicas de Spark DataFrame

In [5]:
# Estadísticas descriptivas de todas las columnas numéricas
df.describe().show()

# Filtrar: solo accidentes con fallecidos > 0 (equivale a WHERE en SQL)
accidentes_fatales = df.filter(df.fallecidos > 0)
print(f"Accidentes fatales: {accidentes_fatales.count():,}")
print(f"Porcentaje: {accidentes_fatales.count()/df.count()*100:.1f}%")

# Seleccionar columnas específicas (equivale a SELECT en SQL)
muestra = df.select("departamento", "tipo_accidente", "fallecidos", "danio_material_soles")
muestra.show(10)

+-------+-----------------+------------+--------------+----------------+---------+------------------+------------------+------------------+----------------------+--------------------+
|summary|     ID_accidente|departamento|tipo_accidente| causa_principal|      mes|              hora|        fallecidos|           heridos|vehiculos_involucrados|danio_material_soles|
+-------+-----------------+------------+--------------+----------------+---------+------------------+------------------+------------------+----------------------+--------------------+
|  count|            50000|       50000|         50000|           50000|    50000|             50000|             50000|             50000|                 50000|               50000|
|   mean|          25000.5|        NULL|          NULL|            NULL|     NULL|          11.43538|           0.21906|           3.49342|               2.49976|          4984.22894|
| stddev|14433.90106658626|        NULL|          NULL|            NULL|     NUL

**Pregunta de reflexión 1.2 — ¿`fallecidos > 0` da el mismo resultado que `fallecidos >= 1`? ¿Y `isNotNull()`?**

> **Respuesta:**
> - `df.fallecidos > 0` y `df.fallecidos >= 1` dan **exactamente el mismo resultado**, porque la columna `fallecidos` es de tipo entero y solo tiene valores 0, 1, 2, 3. En enteros, "mayor que 0" es idéntico a "mayor o igual que 1" (no existe ningún valor intermedio como 0.5).
> - Si cambio la condición a `df.fallecidos.isNotNull()`, el resultado sería **distinto**: devolvería TODOS los registros que tengan un valor (incluidos los que tienen 0 fallecidos), porque `isNotNull()` solo verifica que el campo no esté vacío/nulo, no que sea mayor que cero. En este dataset no hay nulos, así que devolvería los 50,000 registros completos.


## PARTE 2 — IMPLEMENTACIÓN DEL PIPELINE MAPREDUCE CON PYSPARK

### Actividad 2.1 — Análisis de accidentes por departamento (equivalente a MapReduce)

In [6]:
from pyspark.sql import functions as F

# ============================================================
# ANÁLISIS 1: Total de accidentes y fallecidos por departamento
# Equivalente MapReduce: MAP emite (departamento, {accidentes:1, fallecidos:N})
#                        REDUCE suma por departamento
# ============================================================

analisis_departamento = df.groupBy("departamento") \
    .agg(
        F.count("*").alias("total_accidentes"),
        F.sum("fallecidos").alias("total_fallecidos"),
        F.sum("heridos").alias("total_heridos"),
        F.avg("danio_material_soles").alias("danio_promedio_soles"),
        F.sum("danio_material_soles").alias("danio_total_soles")
    ) \
    .orderBy("total_accidentes", ascending=False)

print("=== ACCIDENTES POR DEPARTAMENTO ===")
analisis_departamento.show()

# Guardar resultado en DBFS / sistema de archivos (equivalente a guardar en HDFS)
try:
    analisis_departamento.write.mode("overwrite").csv("/FileStore/resultados/analisis_departamento")
    print("Resultado guardado en DBFS: /FileStore/resultados/analisis_departamento")
except Exception as e:
    # En Colab no existe /FileStore, se guarda en /tmp
    analisis_departamento.write.mode("overwrite").csv("/tmp/resultados/analisis_departamento")
    print("Resultado guardado en: /tmp/resultados/analisis_departamento")

=== ACCIDENTES POR DEPARTAMENTO ===
+------------+----------------+----------------+-------------+--------------------+-----------------+
|departamento|total_accidentes|total_fallecidos|total_heridos|danio_promedio_soles|danio_total_soles|
+------------+----------------+----------------+-------------+--------------------+-----------------+
|        Lima|           17608|            3834|        61457|   4951.360801908224|         87183561|
|    Arequipa|            5953|            1302|        20753|   4957.260540903746|         29510572|
| La Libertad|            4984|            1077|        17367|   5032.992576243981|         25084435|
|       Piura|            4053|             909|        14291|   5086.247224278312|         20614560|
|       Cusco|            3525|             780|        12293|   5049.017021276596|         17797785|
|       Junin|            3474|             771|        12353|   4957.134427173287|         17221085|
|      Loreto|            3006|             64

**Punto de verificación 2.1 — ¿Lima aparece primero con ~35%?**

> **Respuesta:** Sí. Lima aparece primero porque se generó con probabilidad 0.35. Verificación: dividiendo el total de accidentes de Lima entre 50,000 debe dar ≈ 35%, lo que confirma que la agregación distribuida es correcta y respeta la distribución con la que se creó el dataset.

### Actividad 2.2 — Análisis temporal: Horas con más accidentes

In [7]:
# ============================================================
# ANÁLISIS 2: Distribución de accidentes por hora del día
# ============================================================

# Clasificar hora en período del día
df_con_periodo = df.withColumn(
    "periodo_dia",
    F.when(F.col("hora").between(6, 11), "Mañana (6-11h)")
     .when(F.col("hora").between(12, 17), "Tarde (12-17h)")
     .when(F.col("hora").between(18, 23), "Noche (18-23h)")
     .otherwise("Madrugada (0-5h)")
)

analisis_temporal = df_con_periodo.groupBy("periodo_dia") \
    .agg(
        F.count("*").alias("total_accidentes"),
        F.sum("fallecidos").alias("fallecidos"),
        F.round(F.avg("danio_material_soles"), 2).alias("danio_promedio")
    ) \
    .orderBy("total_accidentes", ascending=False)

print("=== ACCIDENTES POR PERÍODO DEL DÍA ===")
analisis_temporal.show()

# -----------------------------------
# RESUELTO: agregar la columna "tasa_mortalidad" = (fallecidos / total_accidentes) * 100
# -----------------------------------
analisis_temporal_con_tasa = analisis_temporal.withColumn(
    "tasa_mortalidad",
    F.round(F.col("fallecidos") / F.col("total_accidentes") * 100, 2)
).orderBy("tasa_mortalidad", ascending=False)

print("=== TASA DE MORTALIDAD POR PERÍODO DEL DÍA ===")
analisis_temporal_con_tasa.show()

=== ACCIDENTES POR PERÍODO DEL DÍA ===
+----------------+----------------+----------+--------------+
|     periodo_dia|total_accidentes|fallecidos|danio_promedio|
+----------------+----------------+----------+--------------+
|Madrugada (0-5h)|           12622|      2885|       4953.06|
|  Mañana (6-11h)|           12610|      2735|       4942.59|
|  Tarde (12-17h)|           12465|      2674|       5036.07|
|  Noche (18-23h)|           12303|      2659|       5006.36|
+----------------+----------------+----------+--------------+

=== TASA DE MORTALIDAD POR PERÍODO DEL DÍA ===
+----------------+----------------+----------+--------------+---------------+
|     periodo_dia|total_accidentes|fallecidos|danio_promedio|tasa_mortalidad|
+----------------+----------------+----------+--------------+---------------+
|Madrugada (0-5h)|           12622|      2885|       4953.06|          22.86|
|  Mañana (6-11h)|           12610|      2735|       4942.59|          21.69|
|  Noche (18-23h)|         

**Pregunta de análisis 2.2 — ¿En qué período la tasa de mortalidad es más alta aunque no tenga más accidentes? ¿Por qué?**

> **Respuesta (hipótesis):** La **Madrugada (0-5h)** suele mostrar la tasa de mortalidad más alta (fallecidos/accidentes) aunque NO sea el período con más accidentes. Mi hipótesis es que de madrugada hay menos tráfico, por lo que los vehículos circulan a **mayor velocidad**, y se concentran factores de riesgo como la **fatiga del conductor**, la **menor visibilidad** y una mayor incidencia de **conducción bajo efectos del alcohol**. Así, aunque ocurran menos accidentes en términos absolutos, cada accidente tiende a ser más severo y letal. Es la diferencia entre *frecuencia* (más choques de día) y *severidad* (choques más mortales de noche/madrugada).

### Actividad 2.3 — Análisis multi-dimensión: Causa × Tipo de accidente

In [ ]:
# Registrar el DataFrame como tabla temporal (vista SQL)
df.createOrReplaceTempView("accidentes")

# Usar Spark SQL directamente
resultado_sql = spark.sql("""
    SELECT
        causa_principal,
        tipo_accidente,
        COUNT(*) as total_casos,
        SUM(fallecidos) as total_fallecidos,
        ROUND(SUM(fallecidos) * 100.0 / COUNT(*), 2) as tasa_mortalidad_pct
    FROM accidentes
    WHERE tipo_accidente IN ('Choque', 'Atropello', 'Volcadura')
    GROUP BY causa_principal, tipo_accidente
    HAVING COUNT(*) > 100
    ORDER BY total_fallecidos DESC
    LIMIT 15
""")

print("=== TOP 15: CAUSA x TIPO (solo accidentes frecuentes) ===")
resultado_sql.show(15, truncate=False)

**Pregunta de análisis 2.3 — ¿Spark SQL o DataFrame API? ¿Cuál prefieres y por qué?** (Luego, el mismo análisis con DataFrame API)

> **Respuesta:** Ambos generan el mismo plan de ejecución porque el **Catalyst Optimizer** los optimiza igual, así que en rendimiento son idénticos. Personalmente prefiero **Spark SQL** para análisis exploratorio y consultas con agregaciones y filtros, porque es más legible y fácil de comunicar a personas que ya conocen SQL (como en mi trabajo, donde reviso datos en SQL Server). Para pipelines de transformación programáticos y reutilizables, prefiero la **DataFrame API**, porque se integra mejor con el código Python, permite encadenar transformaciones y es más fácil de testear y parametrizar.

In [ ]:
# Mismo análisis usando SOLO operaciones de DataFrame (sin spark.sql)
resultado_df_api = df \
    .filter(F.col("tipo_accidente").isin("Choque", "Atropello", "Volcadura")) \
    .groupBy("causa_principal", "tipo_accidente") \
    .agg(
        F.count("*").alias("total_casos"),
        F.sum("fallecidos").alias("total_fallecidos")
    ) \
    .withColumn(
        "tasa_mortalidad_pct",
        F.round(F.col("total_fallecidos") * 100.0 / F.col("total_casos"), 2)
    ) \
    .filter(F.col("total_casos") > 100) \
    .orderBy("total_fallecidos", ascending=False) \
    .limit(15)

print("=== MISMO ANALISIS CON DATAFRAME API ===")
resultado_df_api.show(15, truncate=False)

### Actividad 2.4 — Pipeline completo con Medallion Architecture (Bronze → Silver → Gold)

In [ ]:
# ============================================================
# Pipeline completo Medallion Architecture
# Bronze (raw) -> Silver (limpio) -> Gold (analitico)
# ============================================================

# CAPA BRONZE - datos tal como llegan
print("BRONZE: Datos raw cargados")
print(f"  Registros: {df.count():,} | Columnas: {len(df.columns)}")

# CAPA SILVER - datos limpios y tipados
df_silver = df \
    .filter(df.danio_material_soles > 0) \
    .filter(df.hora.between(0, 23)) \
    .withColumn("es_fatal", F.when(df.fallecidos > 0, True).otherwise(False)) \
    .withColumn("costo_total",
                F.col("danio_material_soles") + F.col("heridos") * 2000 +
                F.col("fallecidos") * 50000)

print(f"\nSILVER: Datos limpios")
print(f"  Registros: {df_silver.count():,} | Columnas: {len(df_silver.columns)}")

# Guardar Silver como Parquet (formato columnar)
silver_path = "/FileStore/silver/accidentes_silver"
try:
    df_silver.write.mode("overwrite").parquet(silver_path)
except Exception:
    silver_path = "/tmp/silver/accidentes_silver"
    df_silver.write.mode("overwrite").parquet(silver_path)
print(f"  Guardado como Parquet en {silver_path}")

# CAPA GOLD - datos agregados para analytics
df_gold = df_silver.groupBy("departamento", "mes") \
    .agg(
        F.count("*").alias("accidentes"),
        F.sum("fallecidos").alias("fallecidos"),
        F.sum("heridos").alias("heridos"),
        F.sum("costo_total").alias("costo_total_estimado"),
        F.sum(F.col("es_fatal").cast("int")).alias("accidentes_fatales")
    ) \
    .withColumn("tasa_fatalidad_pct",
                F.round(F.col("accidentes_fatales") / F.col("accidentes") * 100, 2))

print(f"\nGOLD: Tabla analitica")
df_gold.orderBy("costo_total_estimado", ascending=False).show(10)

# Guardar Gold como Parquet particionado por departamento
gold_path = "/FileStore/gold/accidentes_gold"
try:
    df_gold.write.mode("overwrite").partitionBy("departamento").parquet(gold_path)
except Exception:
    gold_path = "/tmp/gold/accidentes_gold"
    df_gold.write.mode("overwrite").partitionBy("departamento").parquet(gold_path)
print(f"  Guardado particionado por departamento en {gold_path}")

In [ ]:
# Punto de verificacion 2.4: Verificar que los archivos existen
# En Databricks usa dbutils.fs.ls; en Colab usamos os
try:
    display(dbutils.fs.ls("/FileStore/gold/accidentes_gold"))
except NameError:
    import os
    base = gold_path
    print(f"Contenido de {base}:")
    for item in os.listdir(base):
        print("  ", item)

## PARTE 3 — DESAFÍO

### Actividad 3.1 — Window Functions: peor mes por departamento

In [ ]:
# ============================================================
# Window Functions: para cada departamento, el mes con MAS accidentes
# comparado con el promedio mensual del departamento
# ============================================================
from pyspark.sql.window import Window
from pyspark.sql import functions as F

windowDep = Window.partitionBy("departamento")

# 1. Agrupar por departamento + mes
df_mensual = df.groupBy("departamento", "mes") \
    .agg(F.count("*").alias("accidentes_mes"))

# 2. Agregar columnas de ventana: maximo y promedio por departamento
df_con_ventana = df_mensual \
    .withColumn("max_mes", F.max("accidentes_mes").over(windowDep)) \
    .withColumn("promedio_mensual", F.round(F.avg("accidentes_mes").over(windowDep), 1))

# 3. Filtrar solo las filas donde el mes es el maximo del departamento
peor_mes_por_dep = df_con_ventana \
    .filter(F.col("accidentes_mes") == F.col("max_mes")) \
    .withColumn("desviacion_vs_promedio",
                F.round(F.col("accidentes_mes") - F.col("promedio_mensual"), 1)) \
    .select("departamento", "mes", "accidentes_mes", "promedio_mensual", "desviacion_vs_promedio") \
    .orderBy("accidentes_mes", ascending=False)

print("=== PEOR MES POR DEPARTAMENTO (vs su promedio mensual) ===")
peor_mes_por_dep.show(truncate=False)

### Actividad 3.2 — Visualización con Matplotlib

In [ ]:
import matplotlib.pyplot as plt

# Convertir resultado de Spark a Pandas para graficar
df_viz = analisis_departamento.toPandas()

# Grafico de barras horizontales
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_viz['departamento'], df_viz['total_accidentes'], color='steelblue')
ax.set_xlabel('Total de Accidentes')
ax.set_title('Accidentes de Transito por Departamento\nPeru 2023 (Simulado con datos MTC)')
ax.invert_yaxis()  # Mayor en la parte superior
plt.tight_layout()

# Guardar y mostrar
try:
    plt.savefig('/tmp/accidentes_por_departamento.png', dpi=150)
    display(fig)   # En Databricks
except NameError:
    plt.savefig('/tmp/accidentes_por_departamento.png', dpi=150)
    plt.show()     # En Colab
print("Grafico guardado en /tmp/accidentes_por_departamento.png")

**Pregunta de reflexión final 3.1 — ¿Insight inesperado? ¿Y si fueran 2 millones de registros reales? ¿Cambiaría el código?**

> **Respuesta:** Un insight interesante es que el período con más accidentes (frecuencia) no es necesariamente el de mayor tasa de mortalidad (severidad): la madrugada concentra los accidentes más letales aunque ocurran menos.
>
> Si los datos fueran 2 millones de accidentes reales en lugar de 50,000 sintéticos, el **código PySpark NO necesitaría cambios**. Esa es justamente la ventaja de Spark: el mismo código escala de miles a millones (o miles de millones) de registros porque el motor distribuye el trabajo automáticamente entre las particiones/nodos. Lo único que cambiaría es el **tiempo de ejecución** y la necesidad de un **clúster con más nodos**; la lógica de groupBy, agregaciones y window functions se mantiene idéntica. Esto contrasta con pandas, que se quedaría sin memoria con 2 millones de filas pesadas.

**Pregunta de conexión 3.2 — Pacífico Seguros: 3 KPIs para ajustar el SOAT por departamento y datos adicionales necesarios**

> **Respuesta:** Si trabajara en Pacífico Seguros con estos datos del MTC, calcularía estos 3 KPIs por departamento para ajustar las tarifas del SOAT:
>
> 1. **Tasa de siniestralidad por departamento** = accidentes / parque vehicular del departamento. Indica qué tan probable es un accidente donde circula cada vehículo (a mayor tasa, prima más alta).
> 2. **Severidad promedio del siniestro** = costo_total_estimado / número de accidentes. Mide cuánto cuesta en promedio cada accidente (víctimas + daños), clave para dimensionar la prima.
> 3. **Tasa de fatalidad** = accidentes_fatales / total_accidentes. Los fallecidos disparan los pagos del SOAT (cobertura por muerte), así que un departamento con alta fatalidad necesita prima mayor.
>
> **Datos adicionales que necesitaría y NO están en el dataset:**
> - Parque vehicular registrado por departamento (denominador para la siniestralidad)
> - Datos del asegurado: edad del conductor, antigüedad del vehículo, historial de siniestros previos
> - Montos reales pagados por siniestro (no estimados)
> - Kilómetros recorridos / nivel de exposición de cada vehículo
> - Tipo de vía donde ocurrió (urbana, carretera, trocha)

## CONCLUSIÓN DEL LABORATORIO

En este laboratorio implementé un **pipeline Big Data completo con PySpark**:

| Parte | Lo que practiqué |
|-------|------------------|
| Parte 1 | Carga de CSV en Spark, exploración del schema, SparkUI (stages Map/Reduce), filtros y selects |
| Parte 2 | Agregaciones tipo MapReduce (groupBy), análisis temporal, Spark SQL vs DataFrame API, y la **Medallion Architecture** (Bronze → Silver → Gold) con guardado en Parquet |
| Parte 3 | **Window Functions** para el peor mes por departamento y visualización con Matplotlib |

El aprendizaje central es que **el mismo código PySpark escala de 50,000 a millones de registros sin cambios**, porque Spark distribuye el trabajo automáticamente. Esto conecta directamente con el proyecto de mi grupo (**g1-churn-telecom-b2b**), donde aplicaré estas mismas operaciones distribuidas (groupBy, agregaciones, feature engineering) para predecir la fuga de clientes en telecomunicaciones.
